In [9]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(r"D:\孙阳琪简历及复试\孙阳琪简历-找工作版本\已工作版本\孙阳琪_个人简历\projects\面试准备\异乡好居\ai_customer_service.db")

def query(sql):
    return pd.read_sql_query(sql, conn)

# 异乡好居 · 数据分析师(AI方向) · SQL面试真题

> 模拟数据库：`ai_customer_service.db`  |  5张表

## 表结构 & 字段说明

### users（用户表）
| 字段 | 中文含义 | 说明 |
|------|---------|------|
| user_id | 用户ID | 主键 |
| user_name | 用户姓名 | |
| user_type | 用户类型 | new=新用户, returning=老用户 |
| region | 所在地区 | 中国大陆/美国/英国/澳大利亚/欧洲其他 |
| source | 注册渠道 | 搜索/社媒/推荐/广告 |
| register_date | 注册日期 | |

### conversations（AI客服会话表）
| 字段 | 中文含义 | 说明 |
|------|---------|------|
| session_id | 会话ID | 主键 |
| user_id | 用户ID | 外键，关联users表 |
| dt | 会话日期 | |
| intent_category | 意图分类 | 房源咨询/租房流程/支付问题/退改政策/维修报修/签证咨询/其他 |
| turn_count | 对话轮数 | 用户与AI来回对话的轮次 |
| ai_handled | 是否AI处理 | 1=AI处理, 0=纯人工处理 |
| transfer_to_human | 是否转人工 | 1=转人工, 0=未转 |
| response_time_sec | AI响应时间(秒) | AI每条回复的平均耗时 |
| user_satisfaction | 用户满意度 | 1~5分，5=最满意 |
| resolved | 是否解决 | 1=已解决, 0=未解决 |
| experiment_group | AB实验分组 | A=对照组(旧版AI), B=实验组(新版AI) |
| language | 对话语言 | zh=中文, en=英文 |

### human_tickets（人工工单表）
| 字段 | 中文含义 | 说明 |
|------|---------|------|
| ticket_id | 工单ID | 主键 |
| session_id | 关联会话ID | 外键，关联conversations表 |
| agent_id | 客服ID | 处理该工单的人工客服编号 |
| handle_time_min | 处理时长(分钟) | 人工客服处理该工单耗时 |
| resolution_status | 解决状态 | 已解决/未解决/已转交 |

### daily_cost（每日API成本表）
| 字段 | 中文含义 | 说明 |
|------|---------|------|
| dt | 日期 | 主键 |
| total_sessions | 总会话量 | 当天所有会话数 |
| ai_sessions | AI会话量 | 当天AI处理的会话数 |
| api_calls | API调用次数 | 当天调用大模型API的总次数 |
| total_tokens | Token消耗总量 | 当天消耗的Token总数 |
| api_cost_cny | API费用(元) | 当天API调用总费用(人民币) |

### intent_category（意图分类字典表）
| 字段 | 中文含义 | 说明 |
|------|---------|------|
| category_id | 分类ID | 主键 |
| category_name | 分类名称 | 房源咨询/租房流程/支付问题/退改政策/维修报修/签证咨询/其他 |
| description | 分类描述 | 该类意图的具体说明 |

---

### 表关系速查
```
users.user_id  ──→  conversations.user_id
conversations.session_id  ──→  human_tickets.session_id
```

---
## 第1题 · 基础聚合 ⭐⭐

> 统计 **2026年5月** AI处理(`ai_handled=1`)的会话中，各**意图分类**(`intent_category`)的：会话量、平均满意度、转人工率(%)。按会话量降序排列。

In [47]:
# 你的答案
query("""
select
intent_category,
count(distinct session_id) as session_cnt,
round(avg(user_satisfaction),1) as avg_sts,
round(sum(case when transfer_to_human = 1 then 1 else 0 end)  * 100.0 / count(distinct session_id),1) as transfer_rate
from conversations
where dt between '2026-05-01' and '2026-05-31' 
      and ai_handled = 1
group by intent_category
order by session_cnt DESC
""")

,intent_category,session_cnt,avg_sts,transfer_rate
0,房源咨询,115,4.0,19.1
1,租房流程,75,3.8,14.7
2,支付问题,58,4.1,15.5
3,退改政策,37,4.0,13.5
4,维修报修,32,3.4,18.8
5,签证咨询,31,3.8,22.6
6,其他,27,4.1,7.4


In [19]:
# 答案
### 第1题
query("""
SELECT intent_category,
       COUNT(*) AS total_sessions,
       ROUND(AVG(user_satisfaction), 1) AS avg_satisfaction,
       ROUND(SUM(CASE WHEN transfer_to_human = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS transfer_rate
FROM conversations
WHERE ai_handled = 1
  AND dt BETWEEN '2026-05-01' AND '2026-05-31'
GROUP BY intent_category
ORDER BY total_sessions DESC;
""")

,intent_category,total_sessions,avg_satisfaction,transfer_rate
0,房源咨询,115,4.0,19.1
1,租房流程,75,3.8,14.7
2,支付问题,58,4.1,15.5
3,退改政策,37,4.0,13.5
4,维修报修,32,3.4,18.8
5,签证咨询,31,3.8,22.6
6,其他,27,4.1,7.4


---
## 第2题 · 多表关联 ⭐⭐

> 找出所有**转人工且人工最终状态为"未解决"**的会话。显示：`session_id`、`user_name`、`intent_category`、`handle_time_min`。

In [165]:
# 你的答案
query("""
with a as (select
c.session_id,
c.user_id,
c.intent_category,
h.handle_time_min
from conversations c
left join human_tickets h
on c.session_id=h.session_id
where c.transfer_to_human = 1 and h.resolution_status = '未解决')

      
select
a.session_id,
u.user_name,
a.intent_category,
a.handle_time_min
from a
left join users u
on a.user_id=u.user_id
order by a.session_id
""")

,session_id,user_name,intent_category,handle_time_min
0,8,彭博,租房流程,110
1,16,孙强,租房流程,115
2,22,王浩,其他,35
3,31,郑悦,房源咨询,96
4,33,施雨,房源咨询,53
5,39,彭博,签证咨询,76
6,42,韩露,房源咨询,112
7,74,魏岚,房源咨询,97
8,87,褚思,其他,87
9,109,秦岭,签证咨询,75


In [49]:
query(
    """
SELECT c.session_id, u.user_name, c.intent_category, h.handle_time_min
FROM conversations c
JOIN human_tickets h ON c.session_id = h.session_id
JOIN users u ON c.user_id = u.user_id
WHERE c.transfer_to_human = 1
  AND h.resolution_status = '未解决';


"""
)

,session_id,user_name,intent_category,handle_time_min
0,8,彭博,租房流程,110
1,16,孙强,租房流程,115
2,22,王浩,其他,35
3,31,郑悦,房源咨询,96
4,33,施雨,房源咨询,53
5,39,彭博,签证咨询,76
6,42,韩露,房源咨询,112
7,74,魏岚,房源咨询,97
8,87,褚思,其他,87
9,109,秦岭,签证咨询,75


---
## 第3题 · 分组TopN ⭐⭐⭐

> 每个**意图分类**中，找出**响应时间最快的前3条**会话。显示：`intent_category`、`session_id`、`response_time_sec`、排名。

In [53]:
# 你的答案
query("""
select 
intent_category,
session_id,
response_time_sec,
ranking
from (
    select
    intent_category,
    session_id,
    response_time_sec,
    ROW_NUMBER() OVER (PARTITION BY intent_category order by response_time_sec) as ranking
    from conversations
) t
where ranking <=3

""")

,intent_category,session_id,response_time_sec,ranking
0,其他,235,1.6,1
1,其他,32,1.7,2
2,其他,148,2.5,3
3,房源咨询,240,0.8,1
4,房源咨询,282,0.8,2
5,房源咨询,401,0.9,3
6,支付问题,298,0.9,1
7,支付问题,417,0.9,2
8,支付问题,180,1.0,3
9,租房流程,464,0.8,1


---
## 第4题 · AB实验效果对比 ⭐⭐⭐⭐

> `experiment_group` 为 A(对照组-旧版AI) / B(实验组-新版AI)。比较两组在 **2026年5月** 的整体表现：会话量、转人工率(%)、平均满意度、AI解决率(`resolved=1` 且 `transfer_to_human=0`)。

In [54]:
# 你的答案
query("""
select
    a.experiment_group,
    sum(b.total_sessions) as `会话量`,
    round(sum(case when transfer_to_human = 1 then 1 else 0 end) * 100.0 / count(*), 1) as `转人工率(%)`,
    round(avg(user_satisfaction), 1) as `平均满意度`,
    round(sum(case when resolved=1 and transfer_to_human=0 then 1 else 0 end) * 100.0 / count(*), 1) as `AI解决率(%)`
from conversations a
left join daily_cost b
on a.dt=b.dt
and a.dt between '2026-05-01' and '2026-05-31'
group by a.experiment_group


""")

,experiment_group,会话量,转人工率(%),平均满意度,AI解决率(%)
0,A,12255,22.4,4.0,54.5
1,B,12506,12.6,3.9,69.2


---
## 第5题 · 高于均值的用户 ⭐⭐⭐

> 找出**会话次数高于全体用户平均水平**的用户。显示：`user_name`、`region`、会话次数、全体均值。按会话次数降序。

In [57]:
# 你的答案
query("""
select
b.user_name,
b.region,
a.turn_count as `会话次数`,
a.avg_turn_cnt as `全体均值`
from 
    (
    select  
    distinct user_id,
    turn_count,
    round(avg(turn_count) over (partition by user_id),2) as avg_turn_cnt
    from conversations
    ) a
left join users b
on a.user_id=b.user_id
where a.turn_count > a.avg_turn_cnt
order by `会话次数`

""")

,user_name,region,会话次数,全体均值
0,孔琳,澳大利亚,4,3.57
1,马超,中国大陆,4,3.43
2,周晓,中国大陆,5,4.67
3,袁浩,欧洲其他,5,4.79
4,赵明,英国,6,5.67
...,...,...,...,...
157,柏楠,澳大利亚,12,7.30
158,窦鹏,欧洲其他,12,7.83
159,潘悦,美国,12,7.92
160,任杰,中国大陆,12,5.78


---
## 第6题 · CTE + 超均值分析 ⭐⭐⭐⭐

> 用CTE计算每个**意图分类**的平均处理时长（`response_time_sec * turn_count`近似），找出**高于本分类均值50%以上**的会话。显示：`session_id`、`intent_category`、实际时长、分类均值、超出比例(%)。

In [151]:
# 你的答案
query("""
with t as (
select
intent_category,
avg(response_time_sec * turn_count) as avg_time
from conversations
where ai_handled = 1
group by intent_category
)
      
select
c.session_id,
c.intent_category,
(c.response_time_sec * c.turn_count) as actual_time,
t.avg_time,
round((c.response_time_sec * c.turn_count - t.avg_time) * 100.0 / t.avg_time, 1) as exceed_frac
from conversations c
left join t
on c.intent_category=t.intent_category
where (c.response_time_sec * c.turn_count) > t.avg_time * 1.5
ORDER BY exceed_frac DESC;

""")

,session_id,intent_category,actual_time,avg_time,exceed_frac
0,317,支付问题,79.2,19.785246,300.3
1,51,房源咨询,93.6,23.764179,293.9
2,52,支付问题,74.8,19.785246,278.1
3,460,支付问题,73.2,19.785246,270.0
4,454,房源咨询,82.8,23.764179,248.4
...,...,...,...,...,...
115,219,维修报修,36.0,23.451515,53.5
116,153,租房流程,46.8,30.589535,53.0
117,482,其他,51.7,34.092857,51.6
118,393,房源咨询,36.0,23.764179,51.5


In [ ]:
query(
    """
WITH cat_avg AS (
    SELECT intent_category,
           AVG(response_time_sec * turn_count) AS avg_duration
    FROM conversations
    WHERE ai_handled = 1
    GROUP BY intent_category
)
SELECT c.session_id, c.intent_category,
       ROUND(c.response_time_sec * c.turn_count, 1) AS actual_duration,
       ROUND(ca.avg_duration, 1) AS cat_avg_duration,
       ROUND((c.response_time_sec * c.turn_count - ca.avg_duration) / ca.avg_duration * 100, 1) AS exceed_pct
FROM conversations c
JOIN cat_avg ca ON c.intent_category = ca.intent_category
WHERE c.response_time_sec * c.turn_count > ca.avg_duration * 1.5
ORDER BY exceed_pct DESC;
"""
)

,session_id,intent_category,actual_duration,cat_avg_duration,exceed_pct
0,317,支付问题,79.2,19.8,300.3
1,51,房源咨询,93.6,23.8,293.9
2,52,支付问题,74.8,19.8,278.1
3,460,支付问题,73.2,19.8,270.0
4,454,房源咨询,82.8,23.8,248.4
...,...,...,...,...,...
115,219,维修报修,36.0,23.5,53.5
116,153,租房流程,46.8,30.6,53.0
117,482,其他,51.7,34.1,51.6
118,393,房源咨询,36.0,23.8,51.5


---
## 第7题 · LAG环比分析 ⭐⭐⭐⭐

> 按天统计AI处理会话量，用 `LAG()` 计算**前一天会话量**和**日环比变化率(%)**。显示近14天。

In [155]:
# 你的答案
query("""
with a as (
select
dt,
count(*) as ai_cnt
from conversations
where ai_handled=1
group by dt
)
      
select
a.dt,
a.ai_cnt,
lag(a.ai_cnt) over(order by dt) as yst_ai_cnt,
round((a.ai_cnt - lag(a.ai_cnt) over(order by dt)) *100.0/ lag(a.ai_cnt) over(order by dt), 1) as rate
from a
order by a.dt DESC
LIMIT 14


""")

,dt,ai_cnt,yst_ai_cnt,rate
0,2026-06-03,14,16,-12.5
1,2026-06-02,16,12,33.3
2,2026-06-01,12,16,-25.0
3,2026-05-31,16,16,0.0
4,2026-05-30,16,10,60.0
5,2026-05-29,10,16,-37.5
6,2026-05-28,16,11,45.5
7,2026-05-27,11,11,0.0
8,2026-05-26,11,11,0.0
9,2026-05-25,11,11,0.0


In [156]:
query(
"""

WITH daily AS (
    SELECT dt, COUNT(*) AS ai_sessions
    FROM conversations
    WHERE ai_handled = 1
    GROUP BY dt
)
SELECT dt, ai_sessions,
       LAG(ai_sessions) OVER (ORDER BY dt) AS prev_day,
       ROUND((ai_sessions - LAG(ai_sessions) OVER (ORDER BY dt)) * 100.0
             / LAG(ai_sessions) OVER (ORDER BY dt), 1) AS change_pct
FROM daily
ORDER BY dt DESC
LIMIT 14;
"""
    
)

,dt,ai_sessions,prev_day,change_pct
0,2026-06-03,14,16,-12.5
1,2026-06-02,16,12,33.3
2,2026-06-01,12,16,-25.0
3,2026-05-31,16,16,0.0
4,2026-05-30,16,10,60.0
5,2026-05-29,10,16,-37.5
6,2026-05-28,16,11,45.5
7,2026-05-27,11,11,0.0
8,2026-05-26,11,11,0.0
9,2026-05-25,11,11,0.0


---
## 第8题 · 滚动窗口 + 累计求和 ⭐⭐⭐⭐⭐

> 基于 `daily_cost` 表，按天统计：当日API费用(`api_cost_cny`)、截至当天的**累计费用**、**过去7天滚动均值**。

In [157]:
# 你的答案
query("""
select
dt,
api_cost_cny,
sum(api_cost_cny) over (order by dt rows between unbounded preceding and current row) as `截至当天累计`,
round(avg(api_cost_cny) over(order by dt rows between 6 preceding and current row), 1) as `过去7天滚动均值`
from daily_cost
      

      
""")

,dt,api_cost_cny,截至当天累计,过去7天滚动均值
0,2026-05-05,5.56,5.56,5.6
1,2026-05-06,2.75,8.31,4.2
2,2026-05-07,6.62,14.93,5.0
3,2026-05-08,7.32,22.25,5.6
4,2026-05-09,5.10,27.35,5.5
5,2026-05-10,7.83,35.18,5.9
6,2026-05-11,11.77,46.95,6.7
7,2026-05-12,5.61,52.56,6.7
8,2026-05-13,15.44,68.00,8.5
9,2026-05-14,6.22,74.22,8.5


In [104]:
query(
    """
SELECT dt, api_cost_cny,
       SUM(api_cost_cny) OVER (ORDER BY dt
           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cumulative_cost,
       ROUND(AVG(api_cost_cny) OVER (ORDER BY dt
           ROWS BETWEEN 6 PRECEDING AND CURRENT ROW), 2) AS rolling_7d_avg
FROM daily_cost
ORDER BY dt;
"""
)

,dt,api_cost_cny,cumulative_cost,rolling_7d_avg
0,2026-05-05,5.56,5.56,5.56
1,2026-05-06,2.75,8.31,4.15
2,2026-05-07,6.62,14.93,4.98
3,2026-05-08,7.32,22.25,5.56
4,2026-05-09,5.10,27.35,5.47
5,2026-05-10,7.83,35.18,5.86
6,2026-05-11,11.77,46.95,6.71
7,2026-05-12,5.61,52.56,6.71
8,2026-05-13,15.44,68.00,8.53
9,2026-05-14,6.22,74.22,8.47


---
## 第9题 · 用户7日留存 ⭐⭐⭐⭐

> 统计**首次使用AI客服后，7天内再次使用的用户比例**（7日留存率）。思路：①每个用户首次会话日期 → ②7天内有新会话的用户 → ③计算比例。


In [162]:
# 你的答案
query("""
with first_use as (
select
    user_id,
    min(dt) as first_dt
from conversations  
group by user_id    
),
return_use as (   
select 
    distinct f.user_id
from first_use f
left join conversations c
on f.user_id=c.user_id   
where c.dt>f.first_dt
      and c.dt <= date(f.first_dt, '+7 days') 
) 

select
    count(f.user_id) as first_cnt,
    count(r.user_id) as liucun_cnt,
    round(count(r.user_id) * 100.0 / count(f.user_id), 1) as rate
from first_use f
left join return_use r
on f.user_id=r.user_id


""")

,first_cnt,liucun_cnt,rate
0,50,47,94.0


In [121]:
query(
"""
WITH first_use AS (
    SELECT user_id, MIN(dt) AS first_dt
    FROM conversations
    GROUP BY user_id
),
return_users AS (
    SELECT DISTINCT f.user_id
    FROM first_use f
    JOIN conversations c ON f.user_id = c.user_id
    WHERE c.dt > f.first_dt
      AND c.dt <= DATE(f.first_dt, '+7 days')
)
SELECT COUNT(DISTINCT f.user_id) AS total_users,
       COUNT(DISTINCT r.user_id) AS retained_users,
       ROUND(COUNT(DISTINCT r.user_id) * 100.0 / COUNT(DISTINCT f.user_id), 1) AS retention_7d_pct
FROM first_use f
LEFT JOIN return_users r ON f.user_id = r.user_id;
"""

)

,total_users,retained_users,retention_7d_pct
0,50,47,94.0


---
## 第10题 · 综合-用户分层 ⭐⭐⭐⭐⭐

> 将用户按**会话次数**分层：1次→'新用户'、2~5次→'活跃用户'、6次及以上→'重度用户'。统计每层：用户数、占比(%)、**平均满意度**、**转人工率(%)**。


In [125]:
# 你的答案
query("""
with a as (select
user_id,
count(*) as `会话次数`,
(case when count(*) <= 1 then '新用户'
    when count(*) <= 5 then '活跃用户'
    else '重度用户'
end) as `活跃类型`,
avg(user_satisfaction) as `满意度`,
sum(case when transfer_to_human = 1 then 1 else 0 end) as `转人工会话次数`
from conversations
group by user_id
)
      

select
`活跃类型`,
count(*) as `用户数`,
round(count(*) * 100.0 / (select count(*) from a), 2) as `占比(%)`,
round(avg(`满意度`), 2) as `平均满意度`,
round(sum(`转人工会话次数`) * 100.0 / sum(`会话次数`), 2) as `转人工率(%)`
from a
group by `活跃类型`
""")

,活跃类型,用户数,占比(%),平均满意度,转人工率(%)
0,活跃用户,2,4.0,4.20,0.00
1,重度用户,48,96.0,3.92,17.79


In [126]:
query(
    """
WITH user_stats AS (
    SELECT user_id,
           COUNT(*) AS session_cnt,
           AVG(user_satisfaction) AS avg_satis,
           SUM(CASE WHEN transfer_to_human = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS transfer_rate
    FROM conversations
    GROUP BY user_id
),
user_tier AS (
    SELECT *,
           CASE WHEN session_cnt = 1 THEN '新用户'
                WHEN session_cnt BETWEEN 2 AND 5 THEN '活跃用户'
                ELSE '重度用户'
           END AS tier
    FROM user_stats
)
SELECT tier,
       COUNT(*) AS user_cnt,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct,
       ROUND(AVG(avg_satis), 2) AS avg_satisfaction,
       ROUND(AVG(transfer_rate), 1) AS avg_transfer_rate
FROM user_tier
GROUP BY tier
ORDER BY MIN(session_cnt);
"""
)

,tier,user_cnt,pct,avg_satisfaction,avg_transfer_rate
0,活跃用户,2,4.0,4.20,0.0
1,重度用户,48,96.0,3.92,17.8


---
## 参考答案（全部做完后再看！）



### 第2题
```sql

```

### 第3题
```sql
SELECT intent_category, session_id, response_time_sec, ranking
FROM (
    SELECT intent_category, session_id, response_time_sec,
           ROW_NUMBER() OVER (PARTITION BY intent_category ORDER BY response_time_sec ASC) AS ranking
    FROM conversations
    WHERE ai_handled = 1
) t
WHERE ranking <= 3;
```

### 第4题
```sql
SELECT experiment_group,
       COUNT(*) AS session_cnt,
       ROUND(SUM(CASE WHEN transfer_to_human = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS transfer_rate,
       ROUND(AVG(user_satisfaction), 2) AS avg_satisfaction,
       ROUND(SUM(CASE WHEN resolved = 1 AND transfer_to_human = 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS ai_resolution_rate
FROM conversations
WHERE dt BETWEEN '2026-05-01' AND '2026-05-31'
  AND ai_handled = 1
GROUP BY experiment_group;
```

### 第5题
```sql
WITH user_sessions AS (
    SELECT user_id, COUNT(*) AS session_cnt
    FROM conversations
    GROUP BY user_id
)
SELECT u.user_name, u.region, us.session_cnt,
       ROUND((SELECT AVG(session_cnt * 1.0) FROM user_sessions), 1) AS overall_avg
FROM user_sessions us
JOIN users u ON us.user_id = u.user_id
WHERE us.session_cnt > (SELECT AVG(session_cnt * 1.0) FROM user_sessions)
ORDER BY us.session_cnt DESC;

